# CropGuard - Baseline Training (Colab T4)

**Runtime > Change runtime type > T4 GPU**, then Runtime > Run all.

Trains ResNet50 on PlantVillage end to end: download -> validate -> split -> train ->
ONNX -> INT8 -> holdout evaluation. About 90 minutes, mostly waiting on cell 5.

The dataset is pulled from HuggingFace here, not uploaded from your machine. The split is
regenerated from `seed: 42` and hash-checked against the local one, so results stay
comparable without moving 2.2GB around.

## 1. Setup

Clones the repo, installs dependencies, and puts the package on the import path.

We deliberately do **not** `pip install -e .`. An editable install registers the package
through a `.pth` file, and `.pth` files are only read when the interpreter starts - so a
notebook kernel that is already running cannot see it, and `import cropguard` fails.
Adding `src/` to `sys.path` is simpler and works immediately.

Safe to re-run: it pulls instead of failing if the repo is already there.

In [ ]:
import os, sys, importlib, subprocess

REPO = '/content/CropGuard'
SRC  = REPO + '/src'

if os.path.isdir(REPO + '/.git'):
    print('repo present - pulling latest')
    !git -C {REPO} pull -q
else:
    !rm -rf {REPO}
    !git clone -q https://github.com/abhinav7289A/CropGuard.git {REPO}

%cd /content/CropGuard
!git log --oneline -1

In [ ]:
# Colab already ships torch with CUDA. Installing '.[train]' would pull a second torch
# build and can silently replace it with a CPU wheel, so we install only what is missing.
!pip install -q pytorch-lightning timm torchmetrics wandb onnx onnxruntime huggingface_hub scikit-learn tqdm pyyaml

In [ ]:
import os, sys, importlib

SRC = '/content/CropGuard/src'
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()

# Every pipeline step below runs as `!python -m cropguard...` in a fresh subprocess,
# which does not inherit sys.path - PYTHONPATH is what makes those work.
os.environ['PYTHONPATH'] = SRC
os.environ['CROPGUARD_DATA_DIR'] = '/content/cropguard-data'
os.environ['PYTHONIOENCODING'] = 'utf-8'

import cropguard, torch
print('cropguard  ', cropguard.__version__, 'from', os.path.dirname(cropguard.__file__))
print('torch      ', torch.__version__, '| CUDA:', torch.cuda.is_available())
!python -c "import cropguard; print('subprocess OK  ', cropguard.__version__)"

assert torch.cuda.is_available(), 'No GPU - Runtime > Change runtime type > T4 GPU'

## 2. Weights & Biases (optional)

Fill both fields to get experiment tracking. Leave blank and training falls back to a local
CSV logger - it works, but leaves you nothing to show. Key: https://wandb.ai/authorize

In [ ]:
import os

WANDB_API_KEY = ''   # <- paste your key
WANDB_ENTITY  = ''   # <- your W&B username

if WANDB_API_KEY:
    os.environ['WANDB_API_KEY'] = WANDB_API_KEY
    if WANDB_ENTITY:
        os.environ['WANDB_ENTITY'] = WANDB_ENTITY
    os.environ.pop('WANDB_MODE', None)
    import wandb; wandb.login(key=WANDB_API_KEY)   # fail here, not 45 min into training
    print('W&B enabled | project: cropguard-mlops')
else:
    os.environ['WANDB_MODE'] = 'disabled'
    print('W&B disabled - CSV logging only')

## 3. Data

Downloads `data.zip` (2.2GB) from HuggingFace, extracts the `color` variant, validates it,
and builds the leaf-grouped split. ~20 minutes.

The split hash must print `MATCH`. If it does not, the data changed upstream and results
will not be comparable to anything measured locally - stop and investigate.

In [ ]:
!python -m cropguard.data.download --config configs/base.yaml

In [ ]:
!python -m cropguard.data.validate --config configs/base.yaml

In [ ]:
!python -m cropguard.data.split --config configs/base.yaml

import hashlib, json
EXPECTED = '9764d8f2eb2046e9ba91a138e21d472bd6a9e512232431b7d62d252c6ea8efba'
actual = hashlib.sha256(open('/content/cropguard-data/splits.json','rb').read()).hexdigest()
print()
print('split hash:', 'MATCH' if actual == EXPECTED else 'MISMATCH -> ' + actual)
print('leakage   :', json.load(open('/content/cropguard-data/split_report.json'))['leakage']['test'])

## 4. Checkpoints on Drive (recommended)

Colab reclaims free sessions without warning. Checkpoints are written every epoch, so with
Drive mounted you can resume; without it you start over.

In [ ]:
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p /content/drive/MyDrive/cropguard/checkpoints
    !ln -sfn /content/drive/MyDrive/cropguard/checkpoints /content/CropGuard/checkpoints
    print('checkpoints -> Drive')
else:
    !mkdir -p /content/CropGuard/checkpoints
    print('checkpoints are local - a session drop loses them')

## 5. Train

~45-60 min for 12 epochs on a T4. `num_workers` is dropped to 2 to match Colab's 2 vCPUs.

**If the session dies:** re-run cells 1-4, then add
`--resume checkpoints/resnet50-baseline/last.ckpt` to the training command.

**If you hit CUDA OOM:** add `  batch_size: 32` under `train:` in the config cell. Mention
it if you report results - it changes the effective learning-rate schedule.

In [ ]:
%%writefile configs/colab_resnet50.yaml
# Same as the baseline, with dataloader workers matched to Colab's 2 vCPUs.
extends: resnet50_baseline.yaml

data:
  num_workers: 2

In [ ]:
# 30-second smoke test - catches config and data errors before the long run.
!python -m cropguard.training.train --config configs/colab_resnet50.yaml --fast-dev-run

In [ ]:
!python -m cropguard.training.train --config configs/colab_resnet50.yaml

## 6. Export to ONNX + INT8

Gated by a PyTorch parity check (max |logit diff| < 1e-3): a silently wrong graph is the
worst failure mode here, so the export raises rather than shipping one.

In [ ]:
import glob
ckpts = sorted(glob.glob('checkpoints/resnet50-baseline/best-*.ckpt'))
assert ckpts, 'No checkpoint - did training finish?'
CKPT = ckpts[-1]
print('exporting', CKPT)

!python -m cropguard.serving.onnx_export --ckpt "{CKPT}" --out models/cropguard.onnx --quantize
!ls -la models/

## 7. Holdout evaluation

Runs both the fp32 and INT8 graphs over the test split. INT8 accuracy on the full holdout
had never been measured - quantisation is only safe to deploy if the drop is negligible.

In [ ]:
!python -m cropguard.evaluation.predict --model models/cropguard.onnx \
    --split test --out artifacts/preds_fp32.npz --model-version resnet50-fp32
!python -m cropguard.evaluation.predict --model models/cropguard.int8.onnx \
    --split test --out artifacts/preds_int8.npz --model-version resnet50-int8

In [ ]:
from cropguard.evaluation.predict import load_predictions
from cropguard.evaluation.hypothesis import compare_models

fp32 = load_predictions('artifacts/preds_fp32.npz')
int8 = load_predictions('artifacts/preds_int8.npz')
acc32, acc8 = fp32['correct'].mean(), int8['correct'].mean()

print(f'fp32 accuracy : {acc32:.4f}')
print(f'int8 accuracy : {acc8:.4f}')
print(f'quantisation drop : {acc32 - acc8:+.4f}')
print()
# A *significant* result here is a reason not to ship INT8, not a curiosity.
print(compare_models(fp32['correct'], int8['correct'], labels=fp32['labels']).summary())

In [ ]:
# macro-F1 is the metric that matters under 36x class imbalance - accuracy hides rare
# diseases entirely.
import json
from sklearn.metrics import f1_score, classification_report

classes = json.load(open('configs/classes.json'))
y_true, y_pred = fp32['labels'], fp32['predictions']
print('accuracy :', (y_true == y_pred).mean())
print('macro-F1 :', f1_score(y_true, y_pred, average='macro'))
print()
print(classification_report(y_true, y_pred, target_names=classes, digits=3, zero_division=0))

## 8. Save artifacts

Both ONNX graphs and the prediction files the A/B comparison will consume.

In [ ]:
!mkdir -p /content/drive/MyDrive/cropguard/artifacts
!cp -v models/cropguard*.onnx /content/drive/MyDrive/cropguard/artifacts/ || true
!cp -v artifacts/preds_*.npz   /content/drive/MyDrive/cropguard/artifacts/ || true
!ls -la /content/drive/MyDrive/cropguard/artifacts/

---
## 9. OPTIONAL - leakage ablation (~45 min)

The most distinctive result this project can produce. Retrains the identical model on a
naive stratified split, where 74.2% of test images share a physical leaf with training.
The accuracy gap is the inflation that split causes.

Expect the naive split to score **higher** - that is the point. It measures memorisation.

**Warning:** this overwrites `splits.json`. Re-run cell 3's split cell afterwards to restore
the grouped split before doing anything else.

In [ ]:
!python -m cropguard.data.split --config configs/base.yaml --strategy stratified

import json
print(json.load(open('/content/cropguard-data/split_report.json'))['leakage']['test'])

In [ ]:
%%writefile configs/colab_resnet50_leaky.yaml
# Identical to the baseline except for the split it trains on.
extends: resnet50_baseline.yaml

experiment_name: resnet50-baseline-naive-split

data:
  num_workers: 2

In [ ]:
!python -m cropguard.training.train --config configs/colab_resnet50_leaky.yaml

In [ ]:
print('grouped-split accuracy :', acc32)
print('naive-split   accuracy : see test_acc from the run above')
print()
print('The gap is the accuracy inflation caused by leaf leakage.')
print()
# NOTE: these are different test sets, so this is a descriptive comparison. McNemar needs
# a shared holdout and does not apply across the two splits.

---
### Next

1. Report the accuracy, macro-F1, and the fp32 vs INT8 numbers.
2. Train the ConvNeXt-Tiny challenger (`configs/convnext_tiny.yaml`, ~2-3 hrs) for the
   first real A/B comparison.
3. Deploy: point `CROPGUARD_MODEL_PATH` at `cropguard.int8.onnx`.